In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [2]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="MohamedRashad/SADA22", 
    repo_type="dataset", local_dir="./SADA22", allow_patterns="*/*.parquet")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 31 files: 100%|██████████| 31/31 [00:45<00:00,  1.48s/it]


'/home/ubuntu/SADA22'

In [3]:
files = glob('SADA22/*/*.parquet')
len(files)

31

In [4]:
df = pd.read_parquet(files[0])
df

,audio,text,cleaned_text,speaker_age,speaker_gender,speaker_dialect
0,{'bytes': b'RIFFd\\\x04\x00WAVEfmt \x10\x00\x0...,دهن العود ما بدهن عود دهن العود ما نبي الله ير...,دهن العود ما بدهن عود دهن العود ما نبي الله ير...,More than 1 speaker اكثر من متحدث,More than 1 speaker اكثر من متحدث,More than 1 speaker اكثر من متحدث
1,{'bytes': b'RIFF\xe4\xd5\x00\x00WAVEfmt \x10\x...,ما نبي ما نبي يا,ما نبي ما نبي يا,Unknown,Unknown,Unknown
2,{'bytes': b'RIFF$\xfa\x00\x00WAVEfmt \x10\x00\...,ما نبي دهن العود ما نبي,ما نبي دهن العود ما نبي,More than 1 speaker اكثر من متحدث,More than 1 speaker اكثر من متحدث,More than 1 speaker اكثر من متحدث
3,{'bytes': b'RIFF\xe4X\x00\x00WAVEfmt \x10\x00\...,دهن العود,دهن العود,Unknown,Unknown,Unknown
4,{'bytes': b'RIFF\xa4\xd9\x00\x00WAVEfmt \x10\x...,يابا ما نبغى عود ما نبغى عود,يابا ما نبغى عود ما نبغى عود,Unknown,Unknown,Unknown
...,...,...,...,...,...,...
8632,{'bytes': b'RIFF\xe4\xb7\x00\x00WAVEfmt \x10\x...,فضلتك على بناتي,فضلتك على بناتي,Adult -- بالغ,Female,Hijazi
8633,{'bytes': b'RIFF$6\x01\x00WAVEfmt \x10\x00\x00...,يمكن بناتي ما أعطيتهم حنان كثر ما أعطيتك,يمكن بناتي ما اعطيتهم حنان كثر ما اعطيتك,Adult -- بالغ,Female,Hijazi
8634,{'bytes': b'RIFFd\x05\x01\x00WAVEfmt \x10\x00\...,وهذه تالية الله,وهذه تالية الله,Adult -- بالغ,Female,Hijazi
8635,{'bytes': b'RIFF\xa4u\x00\x00WAVEfmt \x10\x00\...,يا روز,يا روز,Adult -- بالغ,Male,Hijazi


In [5]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in tqdm(files):
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in range(len(df)):
            t = df['cleaned_text'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}"
            })
        
    return data

In [20]:
# data = multiprocessing(files, loop, cores = 20)

In [7]:
len(data)

244708

In [8]:
data[0]

{'audio_filename': 'SADA22_audio/SADA22-data-train-00002-of-00028_0.mp3',
 'text': 'دهن العود ما بدهن عود دهن العود ما نبي الله يرحم والديك يا بخور ما نبي يا اخوي في واجد بخور زين انت شوف',
 'speaker': 'SADA22_audio'}

In [9]:
with open('SADA22.json', 'w') as fopen:
    json.dump(data, fopen)

In [10]:
audio_files = [d['audio_filename'] for d in data]

with open('SADA22-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [14]:
# !zip -rq SADA22_audio.zip SADA22_audio

In [15]:
# !hf upload malaysia-ai/Multilingual-TTS SADA22_audio.zip --repo-type=dataset

In [18]:
# !zip -rq SADA22_audio_neucodec.zip SADA22_audio_neucodec

In [19]:
# !hf upload malaysia-ai/Multilingual-TTS SADA22_audio_neucodec.zip --repo-type=dataset

In [21]:
import json

with open('SADA22.json') as fopen:
    rows = json.load(fopen)

mapping = {}
for i in tqdm(range(len(rows))):
    mapping[rows[i]['audio_filename']] = i
len(mapping)

100%|██████████| 244708/244708 [00:00<00:00, 2818532.11it/s]


244708

In [22]:
import faiss
import os
import numpy as np
from tqdm import tqdm

data = {}
d = 192
index = faiss.IndexFlatL2(d)

centroids = []

def assign(x, threshold=0.1):
    if len(centroids) == 0:
        centroids.append(x)
        index.add(np.array([x], dtype=np.float32))
        return 0
    
    D, I = index.search(np.array([x], dtype=np.float32), 1)
    if D[0][0] > threshold:
        centroids.append(x)
        index.add(np.array([x], dtype=np.float32))
        return len(centroids)-1
    else:
        return I[0][0]
        
for i in tqdm(range(len(rows))):
    index_ = mapping[rows[i]['audio_filename']]
    v_f = f'SADA22_embedding/{index_}.npy'
    if not os.path.exists(v_f):
        continue
    try:
        v = np.load(v_f)
        data[rows[i]['audio_filename']] = assign(v)
    except Exception as e:
        pass

100%|██████████| 244708/244708 [04:49<00:00, 844.15it/s] 


In [23]:
for i in range(len(rows)):
    s = data[rows[i]['audio_filename']]
    rows[i]['speaker'] = rows[i]['speaker'] + f'_{s}'

In [24]:
from datasets import Dataset

dataset = Dataset.from_list(rows)
dataset[0]

{'audio_filename': 'SADA22_audio/SADA22-data-train-00002-of-00028_0.mp3',
 'text': 'دهن العود ما بدهن عود دهن العود ما نبي الله يرحم والديك يا بخور ما نبي يا اخوي في واجد بخور زين انت شوف',
 'speaker': 'SADA22_audio_0'}

In [25]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'SADA22')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00,  9.60ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  98%|█████████▊| 17.0MB / 17.3MB, 3.69MB/s  
Processing Files (1 / 1): 100%|██████████| 17.3MB / 17.3MB, 3.60MB/s  
Processing Files (1 / 1): 100%|██████████| 17.3MB / 17.3MB, 3.46MB/s  
New Data Upload: 100%|██████████| 17.3MB / 17.3MB, 3.46MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:05<00:00,  5.54s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/b61351a2a65f8339554cb2826063815bfd129397', commit_message='Upload dataset', commit_description='', oid='b61351a2a65f8339554cb2826063815bfd129397', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)